In [ ]:
# ============================================================
# Dunedain ML/NLP Research Engineer Challenge
# Financial Named Entity Recognition on FiNER-ORD
# FINAL COLAB VERSION - safer token cleaning included
# ============================================================

# ============================================================
# Dunedain ML/NLP Research Engineer Challenge
# Improved Financial NER Model on FiNER-ORD
#
# Improvement over first baseline:
# - Uses dslim/bert-base-NER as a stronger NER-pretrained starting point
# - Uses class-weighted loss to handle O-label imbalance
# - Trains longer with best validation model selected automatically
# ============================================================

# ----------------------------
# 1. Install packages
# ----------------------------
!pip install -q datasets transformers evaluate seqeval accelerate pandas scikit-learn

# ----------------------------
# 2. Imports
# ----------------------------
import os
import json
import random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import evaluate
import torch
from torch import nn

from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

# ----------------------------
# 3. Reproducibility
# ----------------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ----------------------------
# 4. Load dataset
# ----------------------------
raw_dataset = load_dataset("gtfintechlab/finer-ord")

print("\nRaw dataset:")
print(raw_dataset)

print("\nExample raw row:")
print(raw_dataset["train"][0])

# ----------------------------
# 5. Label setup
# ----------------------------
# Original dataset labels:
original_label_list = ["O", "PER_B", "PER_I", "LOC_B", "LOC_I", "ORG_B", "ORG_I"]

# Standard BIO format for seqeval:
label_list = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG"]

id2label = {
    0: "O",
    1: "B-PER",
    2: "I-PER",
    3: "B-LOC",
    4: "I-LOC",
    5: "B-ORG",
    6: "I-ORG",
}

label2id = {label: idx for idx, label in id2label.items()}

print("\nLabel mapping:")
print(id2label)

# ----------------------------
# 6. Token cleaning
# ----------------------------
def clean_token(token):
    """
    Makes sure every token is safe for Hugging Face tokenizers.
    Handles None, NaN, empty strings, and non-string values.
    """
    if token is None:
        return None

    try:
        if pd.isna(token):
            return None
    except TypeError:
        pass

    token = str(token).strip()

    if token == "":
        return None

    return token

# ----------------------------
# 7. Rebuild full sentence-level examples
# ----------------------------
def group_tokens_by_sentence(split):
    """
    The raw dataset has one token per row:
      doc_idx, sent_idx, gold_token, gold_label

    This rebuilds:
      tokens = [token1, token2, ...]
      ner_tags = [label1, label2, ...]
    """
    grouped = defaultdict(lambda: {"tokens": [], "ner_tags": []})
    skipped_tokens = 0

    for row in split:
        token = clean_token(row["gold_token"])

        if token is None:
            skipped_tokens += 1
            continue

        label = int(row["gold_label"])
        key = (int(row["doc_idx"]), int(row["sent_idx"]))

        grouped[key]["tokens"].append(token)
        grouped[key]["ner_tags"].append(label)

    examples = {
        "tokens": [],
        "ner_tags": [],
        "doc_idx": [],
        "sent_idx": []
    }

    for (doc_idx, sent_idx) in sorted(grouped.keys()):
        tokens = grouped[(doc_idx, sent_idx)]["tokens"]
        tags = grouped[(doc_idx, sent_idx)]["ner_tags"]

        if len(tokens) == 0:
            continue

        if len(tokens) != len(tags):
            continue

        examples["tokens"].append(tokens)
        examples["ner_tags"].append(tags)
        examples["doc_idx"].append(doc_idx)
        examples["sent_idx"].append(sent_idx)

    print(f"Skipped invalid tokens: {skipped_tokens}")
    return Dataset.from_dict(examples)

sentence_dataset = DatasetDict({
    "train": group_tokens_by_sentence(raw_dataset["train"]),
    "validation": group_tokens_by_sentence(raw_dataset["validation"]),
    "test": group_tokens_by_sentence(raw_dataset["test"])
})

print("\nSentence-level dataset:")
print(sentence_dataset)

print("\nExample sentence:")
print(sentence_dataset["train"][0])

# ----------------------------
# 8. Dataset statistics
# ----------------------------
def get_label_counts(dataset_split):
    counts = Counter()

    for row in dataset_split:
        for label_id in row["ner_tags"]:
            counts[int(label_id)] += 1

    return counts

train_label_counts = get_label_counts(sentence_dataset["train"])

print("\nTrain label counts:")
for label_id, count in sorted(train_label_counts.items()):
    print(f"{id2label[label_id]}: {count}")

# ----------------------------
# 9. Compute class weights
# ----------------------------
# Most tokens are O, so this helps the model pay more attention to PER/LOC/ORG.
# We use square-root inverse frequency so weights are helpful but not too extreme.

total_labels = sum(train_label_counts.values())
num_labels = len(label_list)

class_weights = []

for label_id in range(num_labels):
    count = train_label_counts.get(label_id, 1)
    weight = np.sqrt(total_labels / (num_labels * count))
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("\nClass weights:")
for i, weight in enumerate(class_weights):
    print(f"{id2label[i]}: {weight.item():.4f}")

# ----------------------------
# 10. Choose stronger model
# ----------------------------
# Stronger than distilbert-base-cased because it already has NER knowledge.
MODEL_NAME = "dslim/bert-base-NER"

# Other experiments you can try later:
# MODEL_NAME = "bert-base-cased"
# MODEL_NAME = "ProsusAI/finbert"

OUTPUT_DIR = "./dunedain_finer_improved_bert_ner"

print("\nUsing model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# ----------------------------
# 11. Tokenize and align labels
# ----------------------------
def tokenize_and_align_labels(examples):
    """
    Align word-level labels to tokenizer subwords.
    We train/evaluate only on first subword of each original token.
    Special tokens and continuation subwords are marked -100.
    """
    cleaned_batch_tokens = []
    cleaned_batch_labels = []

    for tokens, labels in zip(examples["tokens"], examples["ner_tags"]):
        new_tokens = []
        new_labels = []

        for token, label in zip(tokens, labels):
            token = clean_token(token)

            if token is None:
                continue

            new_tokens.append(token)
            new_labels.append(int(label))

        cleaned_batch_tokens.append(new_tokens)
        cleaned_batch_labels.append(new_labels)

    tokenized_inputs = tokenizer(
        cleaned_batch_tokens,
        truncation=True,
        max_length=512,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(cleaned_batch_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(int(labels[word_id]))
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

tokenized_dataset = sentence_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=sentence_dataset["train"].column_names
)

print("\nTokenized dataset:")
print(tokenized_dataset)

# ----------------------------
# 12. Load model
# ----------------------------
# ignore_mismatched_sizes=True is important here because dslim/bert-base-NER
# was trained with its own NER label set. We keep its encoder knowledge but
# create a fresh 7-label classifier for this financial NER dataset.

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# ----------------------------
# 13. Evaluation metric
# ----------------------------
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        pred_labels = []
        gold_labels = []

        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                pred_labels.append(label_list[int(pred_id)])
                gold_labels.append(label_list[int(label_id)])

        true_predictions.append(pred_labels)
        true_labels.append(gold_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ----------------------------
# 14. Custom weighted-loss Trainer
# ----------------------------
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = self.class_weights.to(logits.device)

        loss_fct = nn.CrossEntropyLoss(
            weight=weights,
            ignore_index=-100
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

# ----------------------------
# 15. Training setup
# ----------------------------
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

try:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=8,
        weight_decay=0.01,
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        seed=SEED
    )
except TypeError:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=8,
        weight_decay=0.01,
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        seed=SEED
    )

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# ----------------------------
# 16. Train model
# ----------------------------
print("\nStarting improved training...")
trainer.train()

# ----------------------------
# 17. Evaluate validation and test sets
# ----------------------------
print("\nEvaluating on validation set...")
validation_results = trainer.evaluate(tokenized_dataset["validation"])

print("\nEvaluating on test set...")
test_results = trainer.evaluate(tokenized_dataset["test"])

print("\nValidation results:")
print(validation_results)

print("\nTest results:")
print(test_results)

# ----------------------------
# 18. Save model and results
# ----------------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

results_summary = {
    "model_name": MODEL_NAME,
    "approach": "NER-pretrained BERT with class-weighted loss",
    "epochs": 8,
    "learning_rate": 2e-5,
    "batch_size": 16,
    "validation_results": validation_results,
    "test_results": test_results,
    "label_mapping": id2label,
    "class_weights": {
        id2label[i]: float(class_weights[i].item()) for i in range(len(label_list))
    }
}

with open("dunedain_improved_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

results_table = pd.DataFrame([
    {
        "Model": MODEL_NAME,
        "Approach": "NER-pretrained BERT + weighted loss",
        "Epochs": 8,
        "Validation Precision": validation_results.get("eval_precision"),
        "Validation Recall": validation_results.get("eval_recall"),
        "Validation F1": validation_results.get("eval_f1"),
        "Validation Accuracy": validation_results.get("eval_accuracy"),
        "Test Precision": test_results.get("eval_precision"),
        "Test Recall": test_results.get("eval_recall"),
        "Test F1": test_results.get("eval_f1"),
        "Test Accuracy": test_results.get("eval_accuracy"),
    }
])

print("\nResults table:")
display(results_table)

results_table.to_csv("dunedain_improved_results_table.csv", index=False)

# ----------------------------
# 19. Demo prediction
# ----------------------------
def predict_sentence(tokens):
    cleaned_tokens = []

    for token in tokens:
        token = clean_token(token)
        if token is not None:
            cleaned_tokens.append(token)

    inputs = tokenizer(
        cleaned_tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    device = trainer.model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = trainer.model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()

    word_ids = tokenizer(
        cleaned_tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=512
    ).word_ids()

    final_predictions = []
    previous_word_id = None

    for pred_id, word_id in zip(predictions, word_ids):
        if word_id is None:
            continue

        if word_id != previous_word_id:
            final_predictions.append((cleaned_tokens[word_id], id2label[int(pred_id)]))

        previous_word_id = word_id

    return final_predictions

demo_tokens = [
    "Apple", "CEO", "Tim", "Cook", "visited", "India",
    "after", "Microsoft", "announced", "earnings", "."
]

print("\nDemo prediction:")
print(predict_sentence(demo_tokens))

# ----------------------------
# 20. Write final project summary
# ----------------------------
summary_text = f"""
# Dunedain ML/NLP Research Engineer Challenge
## Improved Financial Named Entity Recognition on FiNER-ORD

### Task
The task was to build a model for Financial Named Entity Recognition using the FiNER-ORD dataset. The goal is to identify person, location, and organization entities in English financial news.

### Dataset Format
The dataset is provided as token-level rows with:
- doc_idx
- sent_idx
- gold_token
- gold_label

Because each row represents one token, I reconstructed sentence-level examples by grouping rows with the same doc_idx and sent_idx. Each final example contains a list of tokens and a matching list of BIO-style entity labels.

### Labels
The original dataset labels were:
- O
- PER_B
- PER_I
- LOC_B
- LOC_I
- ORG_B
- ORG_I

For evaluation with seqeval, I mapped these into standard BIO format:
- O
- B-PER
- I-PER
- B-LOC
- I-LOC
- B-ORG
- I-ORG

### Baseline Motivation
I first used DistilBERT as a fast baseline to validate the pipeline. That confirmed the full process worked: sentence reconstruction, label alignment, training, and evaluation.

### Improved Model
For the improved model, I used {MODEL_NAME}, a BERT model already fine-tuned for NER, and adapted it to the financial NER label set. I used ignore_mismatched_sizes=True so that the model could reuse the NER-informed encoder weights while initializing a fresh classifier for this dataset's 7 labels.

### Class Imbalance Handling
The training set is highly imbalanced, with most tokens labeled O. To address this, I used class-weighted cross entropy loss. The class weights were computed using square-root inverse frequency so that rare entity labels receive more weight without making the training unstable.

### Evaluation
I evaluated using entity-level precision, recall, and F1 with seqeval. This is more appropriate than relying only on token accuracy because token accuracy can be inflated by the large number of O tokens.

### Results
Model: {MODEL_NAME}
Approach: NER-pretrained BERT with class-weighted loss
Epochs: 8
Learning rate: 2e-5
Batch size: 16

Validation Precision: {validation_results.get("eval_precision")}
Validation Recall: {validation_results.get("eval_recall")}
Validation F1: {validation_results.get("eval_f1")}

Test Precision: {test_results.get("eval_precision")}
Test Recall: {test_results.get("eval_recall")}
Test F1: {test_results.get("eval_f1")}

Validation Accuracy: {validation_results.get("eval_accuracy")}
Test Accuracy: {test_results.get("eval_accuracy")}

### Experimental Design
The key experimental idea was to move from a general transformer baseline to a model with prior NER knowledge. Since the dataset is financial news, a further useful experiment would be comparing this NER-pretrained model against a finance-domain encoder such as FinBERT.

### Error Analysis and Next Steps
The next step would be manual error analysis by entity type. I would specifically inspect:
- boundary mistakes where the model identifies an entity but misses part of the span
- ORG versus PER confusion around executives and companies
- ORG versus LOC confusion for governments, countries, and institutions
- rare company names or uncommon international entities

Further improvements could include domain-adaptive pretraining on financial news, a CRF decoding layer, hyperparameter tuning, or ensembling a finance-domain encoder with an NER-pretrained encoder.
"""

with open("dunedain_improved_project_summary.md", "w") as f:
    f.write(summary_text)

print("\nSaved files:")
print("- dunedain_improved_results.json")
print("- dunedain_improved_results_table.csv")
print("- dunedain_improved_project_summary.md")
print(f"- model folder: {OUTPUT_DIR}")

print("\n" + "=" * 70)
print("FINAL IMPROVED PROJECT SUMMARY")
print("=" * 70)
print(f"Model: {MODEL_NAME}")
print("Approach: NER-pretrained BERT + class-weighted loss")
print("Epochs: 8")
print(f"Validation F1: {validation_results.get('eval_f1')}")
print(f"Test F1: {test_results.get('eval_f1')}")
print("=" * 70)

GPU available: True
GPU: Tesla T4

Raw dataset:
DatasetDict({
    train: Dataset({
        features: ['gold_label', 'gold_token', 'doc_idx', 'sent_idx'],
        num_rows: 80531
    })
    validation: Dataset({
        features: ['gold_label', 'gold_token', 'doc_idx', 'sent_idx'],
        num_rows: 10233
    })
    test: Dataset({
        features: ['gold_label', 'gold_token', 'doc_idx', 'sent_idx'],
        num_rows: 25957
    })
})

Example raw row:
{'gold_label': 0, 'gold_token': 'Kenyan', 'doc_idx': 0, 'sent_idx': 0}

Label mapping:
{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-LOC', 4: 'I-LOC', 5: 'B-ORG', 6: 'I-ORG'}
Skipped invalid tokens: 1
Skipped invalid tokens: 0
Skipped invalid tokens: 0

Sentence-level dataset:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'doc_idx', 'sent_idx'],
        num_rows: 3262
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'doc_idx', 'sent_idx'],
        num_rows: 402
    })
    test: Dataset({
   

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...


Evaluating on test set...

Validation results:
{'eval_loss': 0.1730969250202179, 'eval_precision': 0.9039087947882736, 'eval_recall': 0.9113300492610837, 'eval_f1': 0.9076042518397383, 'eval_accuracy': 0.9884675527756059, 'eval_runtime': 1.8195, 'eval_samples_per_second': 220.942, 'eval_steps_per_second': 14.29, 'epoch': 8.0}

Test results:
{'eval_loss': 0.34738394618034363, 'eval_precision': 0.7746591820368885, 'eval_recall': 0.8481123792800702, 'eval_f1': 0.8097233864207879, 'eval_accuracy': 0.9815721500443347, 'eval_runtime': 4.3704, 'eval_samples_per_second': 245.971, 'eval_steps_per_second': 15.559, 'epoch': 8.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Results table:


,Model,Approach,Epochs,Validation Precision,Validation Recall,Validation F1,Validation Accuracy,Test Precision,Test Recall,Test F1,Test Accuracy
0,dslim/bert-base-NER,NER-pretrained BERT + weighted loss,8,0.903909,0.91133,0.907604,0.988468,0.774659,0.848112,0.809723,0.981572



Demo prediction:
[('Apple', 'B-ORG'), ('CEO', 'O'), ('Tim', 'B-PER'), ('Cook', 'I-PER'), ('visited', 'O'), ('India', 'B-LOC'), ('after', 'O'), ('Microsoft', 'B-ORG'), ('announced', 'O'), ('earnings', 'O'), ('.', 'O')]

Saved files:
- dunedain_improved_results.json
- dunedain_improved_results_table.csv
- dunedain_improved_project_summary.md
- model folder: ./dunedain_finer_improved_bert_ner

FINAL IMPROVED PROJECT SUMMARY
Model: dslim/bert-base-NER
Approach: NER-pretrained BERT + class-weighted loss
Epochs: 8
Validation F1: 0.9076042518397383
Test F1: 0.8097233864207879


In [ ]:
import json
from google.colab import files

# Upload the notebook you just downloaded
uploaded = files.upload()

filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove widget metadata that GitHub can't render
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

clean_name = "financial_ner_challenge.ipynb"

with open(clean_name, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

files.download(clean_name)